<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [3]</a>'.</span>

## About the analysis


In [1]:
diag_name = "Annual-mean climatological diagnostics"
diag_desc = "Replication of GFDL's Ferret statistical and plotting infrastructure."
diag_lead_author = "Andrew Wittenberg"
contributors = ["Soelem Bhuiyan", "Aparna Radhakrishnan"]

more_info = """
================================================================================
INPUT REQUIREMENTS & DOCUMENTATION
================================================================================
This script evaluates climatological biases by comparing high-resolution climate 
model outputs against observational datasets. It regrids the model to the obs 
grid using bilinear interpolation after applying a native land mask.

Required Input Data:
  1. Model Data: Monthly time-series NetCDF files (e.g., atmos.*.t_surf.nc).
  2. Obs Data: Monthly observational NetCDF file (e.g., OISST).
  3. Static File: Model grid specification file (atmos.static.nc) containing 
                  the 'land_mask' variable.
================================================================================
"""

## Jupyter Notebook Workflow Outline

The analysis is built upon the following procedures:

---

### <a id="section1"></a>Section 1: Prepare Diagnostics Settings
- Set the parameters  
- Set the runtime configurations  

---

### <a id="section2"></a>Section 2: Load Data from Catalog Files
Two tested options available (users can override with documentation):
- Intake-ESM option  
This section also handles retrieving files from TAPE file using dmget utility

---

### <a id="section3"></a>Section 3: Diagnostics Computation
- Step 1. Compute climatology  
- Step 2. Apply land masking  
- Step 3. Regrid  
- Step 4. Spatial slicing  
- Step 5. Statistics computation  

---

### <a id="section4"></a>Section 4: Results Visualization
- Plot specifications  
- Surface temperature bias maps (SPEAR-HI vs OISST)  

---

### <a id="section5"></a>Section 5: Citations and More About this Diagnostics
- TBA


## Section 1: Prepare Diagnostics Setup 

In [2]:
import xarray as xr
import numpy as np
import xesmf as xe
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import time

# Silence the xarray warnings about future updates
xr.set_options(use_new_combine_kwarg_defaults=True)


## General settings

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [3]:
# ==============================================================================
# DEBUG TIMERS
# ==============================================================================
def debug_print(message, t_start=None):
    """Prints debug messages and elapsed time only if DEBUG_MODE is True."""
    if DEBUG_MODE:
        if t_start is not None:
            print(f"⏱️  {message} Took: {time.perf_counter() - t_start:.2f} seconds")
        else:
            print(f"\n{message}")

if DEBUG_MODE:
    total_start = time.perf_counter()

NameError: name 'DEBUG_MODE' is not defined

## Diagnostics settings

- If the mode is `interactive`, the user will manually input the needed info of the POD options, otherwise the info will be read from the default `settings.jsonc` file.

In [ ]:
# Define a mode (leave "prod" for now)
mode = "interactive"

# Verbosity
verbose = True
# --- Debugging --- #TODO fold this into verbosity
DEBUG_MODE = False  # Set to False to silence all timers and step prints

In [ ]:
# ==============================================================================
# CONTROL PANEL (Edit these variables according to your dataset and preferences)
# ==============================================================================

settings_dict = {
    # --- Paths to data sources---
    "CATALOG_PATH": "/home/Ciheim.Brown/48-SPEAR-cat.json",
    "OBS_PATH": "/home/atw/data/reynolds/v2/noaa.oisst.v2/sst_janstart.nc",
    "STATIC_PATH": "/archive/wfc/SPEAR/SPEAR_HI_8/SPEAR_c384_OM4p08_Control_1990_A13/pp/atmos/atmos.static.nc",

    # --- Variables and unit conversions---
    "MODEL_VAR": "t_surf",
    "OBS_VAR": "SST",
    "MODEL_OFFSET": -273.15,

    # --- Metadata ---
    "freq": "mon",
    "chunk_freq": "10yr",
    "realm": "atmos",

    # --- Temporal range ---
    "MODEL_TIME_SLICE": slice(0, 1200),
    "OBS_TIME_SLICE": slice("1982", "2016"),

    # --- Spatial Bounding Box / Domain ---
    "LAT_BOUNDS": [-20, 20],
    "LON_BOUNDS": [25, 385],

    # --- Plotting and Contour levels ---
    "PLOT_TITLE_MODEL": "SPEAR Model (Years 0001-0100)",
    "PLOT_TITLE_OBS": "OISST Observations (1982-2016)",
    "CBAR_LABEL": "SST (°C)",

    "SST_SHADE_LEVELS": np.arange(16, 33.5, 0.5),
    "SST_LINE_LEVELS": np.arange(16, 34, 1.0),
    "BIAS_SHADE_LEVELS": np.arange(-5, 5.25, 0.25),
    "BIAS_LINE_LEVELS": np.arange(-5, 5.5, 0.5),
    "BIAS_TICK_INTERVAL": np.arange(-5, 6, 1)
}

# Convert the settings dictionary into an object so values can be accessed
# using dot notation instead of dictionary keys (e.g., settings.OBS_PATH)
from types import SimpleNamespace
settings = SimpleNamespace(**settings_dict)

## Section 2 Loading Data from Catalogs
lets figure out ds_model with catalogs

In [ ]:
import intake, intake_esm

In [ ]:
cat = intake.open_esm_datastore(settings.CATALOG_PATH)

#Exception - TODO to be addressed at the catalog level
df = cat.df 
df['table_id'] = df['table_id'].fillna('NA')
df['table_id'].value_counts(dropna=False)

cat_subset = cat.search(
    variable_id=settings.MODEL_VAR,
    frequency=settings.freq,
    realm=settings.realm,
    chunk_freq=settings.chunk_freq,
    cell_methods="ts", 
)

In [ ]:
cat_subset

In [ ]:
#dmgets the files before we proceed
#TODO check if the filesystem is TAPE or not
from utils import dmget
dmstatus = cat_subset.df["path"].apply(dmget.dmgetmagic)

In [ ]:
dset_dict = cat_subset.to_dataset_dict(
    xarray_open_kwargs={"decode_times": True}
)
dset_dict.keys()

In [ ]:
#TODO don't hardcode this, sanity check dset_dict and retrieve from that with index
ds_model = dset_dict['SPEAR_HI_8.SPEAR_c384_OM4p08_Control_1990_A13.mon.NA.atmos.10yr']

In [ ]:
ds_model

In [ ]:
# ==========================================
# 1. LOAD DATA
# ==========================================
debug_print("[1/7] Loading data...")
t0 = time.perf_counter()

ds_obs = xr.open_dataset(settings.OBS_PATH)

ds_model = ds_model.isel(time=settings.MODEL_TIME_SLICE)

debug_print("Step 1", t0)

## Section 3 Diagnostics Computation

###1. Computing Climatology

In [ ]:
# ==========================================
# 2. COMPUTING CLIMATOLOGY
# ==========================================
debug_print("[2/7] Computing climatologies...")
t0 = time.perf_counter()

model_clim = ds_model[settings.MODEL_VAR].mean(dim='time', keep_attrs=True)
model_clim = model_clim + settings.MODEL_OFFSET 
obs_clim = ds_obs[settings.OBS_VAR].sel(time=settings.OBS_TIME_SLICE).mean(dim='time', keep_attrs=True)

model_clim = model_clim.compute()
obs_clim = obs_clim.compute()

debug_print("Step 2", t0)

####2.Apply Land Masking

In [ ]:
# ==========================================
# 3. NATIVE LAND MASK
# ==========================================
debug_print("[3/7] Applying native model land mask...")
t0 = time.perf_counter()

try:
    ds_static = xr.open_dataset(settings.STATIC_PATH)
    if 'land_mask' in ds_static:
        model_clim = model_clim.where(ds_static['land_mask'] < 0.5)
    elif 'frac_ocean' in ds_static:
        model_clim = model_clim.where(ds_static['frac_ocean'] > 0.5)
    else:
        print("WARNING: 'land_mask' or 'frac_ocean' not found in static file. Land bleed may occur.")
except FileNotFoundError:
    print(f"WARNING: Could not find {settings.STATIC_PATH}. Land bleed will occur!")

debug_print("Step 3", t0)

####3. Regridding

In [ ]:
# ==========================================
# 4. REGRIDDING
# ==========================================
debug_print("[4/7] Regridding Model to Obs (Bilinear)...")
t0 = time.perf_counter()

regridder = xe.Regridder(model_clim, obs_clim, 'bilinear', periodic=True)
model_regridded = regridder(model_clim)

debug_print("Step 4", t0)

####4. Spatial Slicing

In [ ]:
# ==========================================
# 5. SPATIAL SLICING & MASKING
# ==========================================
debug_print("[5/7] Isolating spatial domain and applying common mask...")
t0 = time.perf_counter()

model_domain = model_regridded.sel(lat=slice(settings.LAT_BOUNDS[0], settings.LAT_BOUNDS[1]))
obs_domain = obs_clim.sel(lat=slice(settings.LAT_BOUNDS[0], settings.LAT_BOUNDS[1]))

common_mask = model_domain.notnull() & obs_domain.notnull()

a_var = obs_domain.where(common_mask)   # Ferret Variable A (Obs)
b_var = model_domain.where(common_mask) # Ferret Variable B (Model)
bma_var = b_var - a_var                 # Ferret Bias (Model minus Obs)

debug_print("Step 5", t0)

####5. Statistics Computation

In [ ]:
# ==========================================
# 6. SPHERICAL STATISTICS (Cosine Weighting)
# ==========================================
debug_print("[6/7] Calculating spherically-weighted statistics...")
t0 = time.perf_counter()

weights = np.cos(np.deg2rad(a_var.lat))
weights.name = "weights"

def get_weighted_stats(da, w):
    daw = da.weighted(w)
    ave = daw.mean(skipna=True).values
    std = daw.std(skipna=True).values
    vmax = da.max(skipna=True).values
    vmin = da.min(skipna=True).values
    return ave, std, vmax, vmin

a_ave, a_std, a_max, a_min = get_weighted_stats(a_var, weights)
b_ave, b_std, b_max, b_min = get_weighted_stats(b_var, weights)
bma_ave, bma_std, bma_max, bma_min = get_weighted_stats(bma_var, weights)

ab_rmsd = np.sqrt(((bma_var)**2).weighted(weights).mean(skipna=True)).values
covar = ((b_var - b_ave) * (a_var - a_ave)).weighted(weights).mean(skipna=True)
ab_corr = (covar / (b_var.weighted(weights).std(skipna=True) * a_var.weighted(weights).std(skipna=True))).values

if DEBUG_MODE:
    print(f"Stats Check -> Bias Ave: {bma_ave:.3f}, RMSD: {ab_rmsd:.3f}, Corr: {ab_corr:.2f}")

debug_print("Step 6", t0)

## Section 4 Results Visualization

In [ ]:
# ==========================================
# 7. EXECUTION & PLOTTING 
# ==========================================
debug_print("[7/7] Plotting data...")
t0 = time.perf_counter()

fig, axes = plt.subplots(3, 1, figsize=(10, 12), subplot_kw={'projection': ccrs.PlateCarree(central_longitude=205)})

plt.subplots_adjust(right=0.75, bottom=0.08, hspace=0.3)

for ax in axes:
    ax.set_extent([settings.LON_BOUNDS[0], settings.LON_BOUNDS[1], settings.LAT_BOUNDS[0], settings.LAT_BOUNDS[1]], crs=ccrs.PlateCarree())
    ax.set_aspect('auto', adjustable='box')
    ax.add_feature(cfeature.LAND, facecolor='black', zorder=2)
    ax.coastlines(color='white', linewidth=0.5, zorder=3)
    ax.set_xticks(np.arange(settings.LON_BOUNDS[0], settings.LON_BOUNDS[1]+1, 60), crs=ccrs.PlateCarree())
    ax.set_yticks(np.arange(settings.LAT_BOUNDS[0], settings.LAT_BOUNDS[1]+1, 10), crs=ccrs.PlateCarree())
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.gridlines(color='black', linestyle='-', alpha=0.3, zorder=4)

# --- Panel 1: Obs ---
cs1 = a_var.plot.contourf(ax=axes[0], transform=ccrs.PlateCarree(), levels=settings.SST_SHADE_LEVELS, cmap='turbo', add_colorbar=False, extend='both')
line1 = a_var.plot.contour(ax=axes[0], transform=ccrs.PlateCarree(), levels=settings.SST_LINE_LEVELS, colors='black', linewidths=0.8)
axes[0].clabel(line1, inline=True, fontsize=8, fmt='%1.0f')
axes[0].set_title(settings.PLOT_TITLE_OBS)
fig.colorbar(cs1, ax=axes[0], label=settings.CBAR_LABEL, pad=0.02, shrink=0.85)

axes[0].text(1.18, 0.5, f"ave\n{a_ave:.1f}\n\nstd\n{a_std:.2f}\n\nmax\n{a_max:.1f}\n\nmin\n{a_min:.1f}", 
             transform=axes[0].transAxes, va='center', ha='center', fontsize=10, family='monospace', clip_on=False)

# --- Panel 2: Model ---
cs2 = b_var.plot.contourf(ax=axes[1], transform=ccrs.PlateCarree(), levels=settings.SST_SHADE_LEVELS, cmap='turbo', add_colorbar=False, extend='both')
line2 = b_var.plot.contour(ax=axes[1], transform=ccrs.PlateCarree(), levels=settings.SST_LINE_LEVELS, colors='black', linewidths=0.8)
axes[1].clabel(line2, inline=True, fontsize=8, fmt='%1.0f')
axes[1].set_title(settings.PLOT_TITLE_MODEL)
fig.colorbar(cs2, ax=axes[1], label=settings.CBAR_LABEL, pad=0.02, shrink=0.85)

axes[1].text(1.18, 0.5, f"ave\n{b_ave:.1f}\n\nstd\n{b_std:.2f}\n\nmax\n{b_max:.1f}\n\nmin\n{b_min:.1f}", 
             transform=axes[1].transAxes, va='center', ha='center', fontsize=10, family='monospace', clip_on=False)

# --- Panel 3: Bias ---
cs3 = bma_var.plot.contourf(ax=axes[2], transform=ccrs.PlateCarree(), levels=settings.BIAS_SHADE_LEVELS, cmap='RdBu_r', add_colorbar=False, extend='both')
line_widths = [1.5 if lev == 0 else 0.8 for lev in settings.BIAS_LINE_LEVELS]
line3 = bma_var.plot.contour(ax=axes[2], transform=ccrs.PlateCarree(), levels=settings.BIAS_LINE_LEVELS, colors='black', linewidths=line_widths)
axes[2].clabel(line3, inline=True, fontsize=8, fmt='%1.1f')
axes[2].set_title("Difference (Model - Obs)")
fig.colorbar(cs3, ax=axes[2], label='Bias (°C)', pad=0.02, shrink=0.85, ticks=settings.BIAS_TICK_INTERVAL)

axes[2].text(1.18, 0.5, f"ave\n{bma_ave:.3f}\n\nstd\n{bma_std:.3f}\n\nmax\n{bma_max:.2f}\n\nmin\n{bma_min:.2f}", 
             transform=axes[2].transAxes, va='center', ha='center', fontsize=10, family='monospace', clip_on=False)

axes[2].text(0.25, -0.25, f"corr(a,b) = {ab_corr:.2f}", transform=axes[2].transAxes, ha='center', fontsize=12, family='monospace', clip_on=False)
axes[2].text(0.75, -0.25, f"RMSD(a,b) = {ab_rmsd:.3f}", transform=axes[2].transAxes, ha='center', fontsize=12, family='monospace', clip_on=False)

# Save Figure
plt.savefig("t_surf_final_replica.png", dpi=600, bbox_inches='tight')
plt.show()

debug_print("Step 7", t0)

if DEBUG_MODE:
    print(f"\n✅ TOTAL PIPELINE TIME: {time.perf_counter() - total_start:.2f} seconds")